# DataMatch: прогноз продаж билетов ХК «Авангард»

Финальный ноутбук. С нуля (после `git clone` и `pip install -r requirements.txt`) одним запуском
выдаёт `predictions.csv` в корне репозитория: прогноз `tickets_total` для 17 test-матчей × 8 зон.

Все данные берутся через `src.data.build_table(as_of)`, дата отсечки 15.12.2025.
Валидация — `src.validation.evaluate`: 3 фолда «псевдо-сегодня» (2023-12-15, 2024-11-25, 2025-09-20).

## Настройки

In [1]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import catboost
import matplotlib
import numpy as np
import pandas as pd
import seaborn
import sklearn

from src.baseline import smart_baseline_predict, zone_mean_predict
from src.data import build_table, load_raw
from src.validation import evaluate, predict_test

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python    ", sys.version.split()[0])
for lib in [pd, np, sklearn, catboost, matplotlib, seaborn]:
    print(f"{lib.__name__:<10}", lib.__version__)

Python     3.12.10
pandas     3.0.6
numpy      2.5.3
sklearn    1.9.1
catboost   1.2.10
matplotlib 3.11.2
seaborn    0.13.2


In [2]:
# Модели по контракту predict_fn(train, val, as_of) -> массив прогнозов (см. src/validation.py).
# Чтобы добавить модель: импортировать её predict_fn и добавить запись, например
#     from src.model import catboost_predict
#     MODELS["catboost"] = catboost_predict
# и переключить MODEL ниже. Больше в ноутбуке ничего менять не нужно.
from src.model import catboost_predict

MODELS = {
    "smart_baseline": smart_baseline_predict,
    "catboost": catboost_predict,
}

MODEL = "catboost"
predict_fn = MODELS[MODEL]
print("Модель:", MODEL)

Модель: catboost


## Данные

In [3]:
df = build_table()
train = df[df["tickets_total"].notna()]
test = df[df["is_test"] == 1]
started = test.groupby("match_id")["sales_started"].max()

print(f"train: {len(train)} строк, {train['match_id'].nunique()} матчей "
      f"({train['match_date'].min().date()} — {train['match_date'].max().date()})")
print(f"test:  {len(test)} строк, {test['match_id'].nunique()} матчей "
      f"({test['match_date'].min().date()} — {test['match_date'].max().date()})")
print(f"test-матчи с начавшимися продажами ({started.sum()}):", list(started[started == 1].index))

train: 680 строк, 85 матчей (2023-09-02 — 2025-12-07)
test:  136 строк, 17 матчей (2025-12-24 — 2026-03-13)
test-матчи с начавшимися продажами (6): ['M086', 'M087', 'M088', 'M089', 'M090', 'M091']


## Валидация

Три модели на одних фолдах: `zone_mean` (baseline из задания — среднее по зоне),
`smart_baseline` (уровень зоны в текущем сезоне + уже проданное) и выбранная `MODEL`.
RMSE общий и по фолдам. Assert'ы ловят изменение валидации (эталонные RMSE baseline'ов)
и проверяют, что выбранная модель лучше smart baseline.

In [4]:
res = {"zone_mean": evaluate(zone_mean_predict), "smart_baseline": evaluate(smart_baseline_predict)}
if MODEL not in res:
    res[MODEL] = evaluate(predict_fn)

rmse_table = pd.concat({name: r.by_fold["rmse"] for name, r in res.items()}, axis=1)
if MODEL != "smart_baseline":
    rmse_table[f"{MODEL} − smart"] = rmse_table[MODEL] - rmse_table["smart_baseline"]
print("RMSE: " + ", ".join(f"{name} = {r.overall:.1f}" for name, r in res.items()))

zone_mean, smart = res["zone_mean"].overall, res["smart_baseline"].overall
assert abs(zone_mean - 141.0) < 0.05, f"zone_mean RMSE {zone_mean:.2f} != 141.0 — валидация изменилась"
assert abs(smart - 135.2) < 0.05, f"smart_baseline RMSE {smart:.2f} != 135.2 — валидация изменилась"
if MODEL == "smart_baseline":
    print("Выбран smart_baseline — сравнение «модель лучше smart» не применяется")
else:
    assert res[MODEL].overall < smart, f"{MODEL} RMSE {res[MODEL].overall:.2f} не лучше smart_baseline {smart:.2f}"

rmse_table.round(1)

RMSE: zone_mean = 141.0, smart_baseline = 135.2, catboost = 124.7


,zone_mean,smart_baseline,catboost,catboost − smart
fold,,,,
2023-12-15,172.2,172.2,163.3,-8.9
2024-11-25,103.0,79.1,83.1,4.0
2025-09-20,147.2,148.5,121.6,-26.9
all,141.0,135.2,124.7,-10.5


## Прогноз

In [5]:
sub = predict_test(predict_fn)
sub.pivot(index="match_id", columns="zone", values="tickets_total")

zone,1,2,3,4,5,6,7,8
match_id,,,,,,,,
M086,450,256,583,889,745,642,777,415
M087,475,266,623,912,811,884,937,440
M088,407,238,514,810,760,753,789,392
M089,472,267,630,882,809,820,851,427
M090,469,287,662,940,870,700,836,467
M091,472,279,651,947,890,652,795,455
M092,482,273,652,826,777,696,810,429
M093,477,267,651,863,792,718,837,436
M094,484,273,660,893,831,760,908,452


## Проверки перед сохранением

In [6]:
def check_submission(sub):
    tmpl = load_raw()["submission"]
    assert len(sub) == 136, f"строк {len(sub)}, нужно 136"
    print("OK 136 строк")
    assert list(sub.columns) == ["match_id", "zone", "tickets_total"]
    assert sub[["match_id", "zone"]].equals(tmpl[["match_id", "zone"]]), "match_id/zone или порядок не как в шаблоне"
    print("OK match_id, zone и порядок как в submission_template.csv")
    assert sub["tickets_total"].notna().all(), "есть пропуски"
    print("OK нет пропусков")
    assert sub["tickets_total"].dtype.kind == "i" and (sub["tickets_total"] >= 0).all(), "не целые или < 0"
    print("OK целые, >= 0")
    bounds = sub.merge(build_table()[["match_id", "zone", "sold_as_of", "capacity"]],
                       on=["match_id", "zone"], how="left", validate="one_to_one")
    assert (bounds["tickets_total"] >= bounds["sold_as_of"]).all(), "прогноз меньше уже проданного"
    assert (bounds["tickets_total"] <= bounds["capacity"]).all(), "прогноз больше capacity"
    print("OK sold_as_of <= прогноз <= capacity")


check_submission(sub)

OK 136 строк
OK match_id, zone и порядок как в submission_template.csv
OK нет пропусков
OK целые, >= 0
OK sold_as_of <= прогноз <= capacity


## Сохранение

In [7]:
path = ROOT / "predictions.csv"
sub.to_csv(path, index=False, encoding="utf-8")

saved = pd.read_csv(path, encoding="utf-8")
check_submission(saved)
print(f"Сохранено: {path.name}, {len(saved)} строк, сумма прогноза {saved['tickets_total'].sum()}")

OK 136 строк
OK match_id, zone и порядок как в submission_template.csv
OK нет пропусков
OK целые, >= 0
OK sold_as_of <= прогноз <= capacity
Сохранено: predictions.csv, 136 строк, сумма прогноза 85631
